# 1.1 Data source profiling

Perfilamiento de la hoja `query` a partir de hipótesis: (1) completitud, (2) duplicados, (3) llave de la fila,
(4) formato de códigos, (5) tipos numéricos, (6) formato y cobertura de `Corte`, (7) consistencia código–nombre.

Artefactos persistidos: ninguno (diagnóstico). Se lee el archivo original como texto para no perder valores por conversión de tipos.

| Artefacto | Ruta |
|---|---|
| Entrada | `data/0_source/KPIS_historico.xlsx` (hoja `query`) |


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

SOURCE_PATH = Path("../../data/0_source/KPIS_historico.xlsx")
LLAVE = ["Corte", "Codigo_EQU", "Indicador"]
PATRON_CODIGO = r"[A-Z]{3}\d{5}"
PATRON_CORTE = r"20\d{2}(0[1-9]|1[0-2])"

HIPOTESIS = []


def registrar_hipotesis(codigo, enunciado, resultado, evidencia):
    """resultado: 'se cumple' | 'no se cumple' | 'parcial'."""
    HIPOTESIS.append({"hipotesis": codigo, "enunciado": enunciado, "resultado": resultado, "evidencia": evidencia})
    print(f"{codigo} | {resultado.upper()} | {evidencia}")


query = pd.read_excel(SOURCE_PATH, sheet_name="query", dtype=str)
print("query | filas:", len(query), "| columnas:", list(query.columns))


query | filas: 15188 | columnas: ['Frente', 'Corte', 'Codigo_EQU', 'EQU', 'Indicador', 'Resultado', 'Meta', 'Cumplimiento']


## H01. Las columnas de `query` no tienen valores nulos


In [2]:
nulos = query.isna().sum().rename("nulos").to_frame()
nulos["pct"] = (nulos.nulos / len(query)).round(4)
display(nulos.T)

con_nulos = nulos[nulos.nulos > 0]
registrar_hipotesis(
    "H01", "Las columnas de query no tienen valores nulos",
    "se cumple" if con_nulos.empty else "no se cumple",
    "; ".join(f"{c}: {int(r.nulos)} ({r.pct:.1%})" for c, r in con_nulos.iterrows()),
)


,Frente,Corte,Codigo_EQU,EQU,Indicador,Resultado,Meta,Cumplimiento
nulos,0.0,0.0,36.0000,6185.0000,0.0,462.0000,308.0000,265.0000
pct,0.0,0.0,0.0024,0.4072,0.0,0.0304,0.0203,0.0174


H01 | NO SE CUMPLE | Codigo_EQU: 36 (0.2%); EQU: 6185 (40.7%); Resultado: 462 (3.0%); Meta: 308 (2.0%); Cumplimiento: 265 (1.7%)


## H02. No hay filas duplicadas


In [3]:
duplicadas = query.duplicated()
por_corte = query[duplicadas].Corte.value_counts().head(3)
registrar_hipotesis(
    "H02", "No hay filas duplicadas",
    "se cumple" if duplicadas.sum() == 0 else "no se cumple",
    f"{duplicadas.sum()} filas exactas repetidas ({duplicadas.mean():.1%}); cortes con más casos: "
    + ", ".join(f"{k}: {v}" for k, v in por_corte.items()),
)


H02 | NO SE CUMPLE | 2182 filas exactas repetidas (14.4%); cortes con más casos: 202502: 1223, 202607: 588, 202603: 92


## H03. Cada fila es única por `Corte × Codigo_EQU × Indicador`

Hipótesis de grano: una fila es la medición mensual de un indicador para un equipo. Se evalúa después de quitar duplicados exactos.


In [4]:
sin_duplicados = query.drop_duplicates()
repetidas = sin_duplicados[sin_duplicados.duplicated(LLAVE, keep=False)]
llaves = repetidas.groupby(LLAVE, dropna=False).ngroups

display(repetidas.groupby(["Corte", "Indicador"]).size().sort_values(ascending=False).head(6).rename("filas").to_frame())
print("¿Frente cambia dentro de una llave?:", sin_duplicados.groupby(LLAVE).Frente.nunique().gt(1).any())
registrar_hipotesis(
    "H03", "Cada fila es única por Corte × Codigo_EQU × Indicador",
    "se cumple" if llaves == 0 else "no se cumple",
    f"{llaves} llaves con valores distintos ({len(repetidas)} filas); 202502 trae Percepción/Adopción por respuesta individual",
)


filas
Corte  Indicador                
202502 Percepción            874
       Adopción               88
202503 Talento + Agilidad     28
202504 Talento + Agilidad     28
202505 Talento + Agilidad     28
202506 Talento + Agilidad     28

¿Frente cambia dentro de una llave?: False
H03 | NO SE CUMPLE | 247 llaves con valores distintos (1292 filas); 202502 trae Percepción/Adopción por respuesta individual


## H04. `Codigo_EQU` sigue un formato estándar (3 letras mayúsculas + 5 dígitos)


In [5]:
codigos = query.Codigo_EQU.dropna()
fuera_formato = codigos[~codigos.str.fullmatch(PATRON_CODIGO)]
print("prefijos:", codigos.str[:3].value_counts().to_dict())
registrar_hipotesis(
    "H04", "Codigo_EQU sigue el formato AAA00000",
    "se cumple" if fuera_formato.empty else "no se cumple",
    f"{len(fuera_formato)} filas fuera de formato; ejemplos: {', '.join(fuera_formato.unique()[:4])}",
)


prefijos: {'EQU': 14056, 'CEX': 772, 'Equ': 324}
H04 | NO SE CUMPLE | 325 filas fuera de formato; ejemplos: Equ00074, Equ00052, Equ00026, Equ00028


## H05. `Resultado`, `Meta` y `Cumplimiento` son numéricos


In [6]:
no_numericos = {}
for col in ["Resultado", "Meta", "Cumplimiento"]:
    valores = pd.to_numeric(query[col], errors="coerce")
    no_numericos[col] = int((query[col].notna() & valores.isna()).sum())
    print(col, "| min:", round(valores.min(), 3), "| mediana:", round(valores.median(), 3), "| max:", round(valores.max(), 3))
registrar_hipotesis(
    "H05", "Resultado, Meta y Cumplimiento son numéricos",
    "se cumple" if sum(no_numericos.values()) == 0 else "no se cumple",
    f"valores no numéricos: {no_numericos}; los rangos (p. ej. Meta hasta 15.6 millones) se analizan por indicador en 1_2",
)


Resultado | min: -423.957 | mediana: 1.016 | max: 57864.0
Meta | min: -1.771 | mediana: 1.0 | max: 15627925.0
Cumplimiento | min: -1872.998 | mediana: 1.099 | max: 155.421
H05 | SE CUMPLE | valores no numéricos: {'Resultado': 0, 'Meta': 0, 'Cumplimiento': 0}; los rangos (p. ej. Meta hasta 15.6 millones) se analizan por indicador en 1_2


## H06. `Corte` tiene formato `YYYYMM` y H07. la cobertura entre cortes es estable


In [7]:
formato_ok = query.Corte.str.fullmatch(PATRON_CORTE).all()
registrar_hipotesis("H06", "Corte tiene formato YYYYMM", "se cumple" if formato_ok else "no se cumple",
                    f"{query.Corte.nunique()} cortes entre {query.Corte.min()} y {query.Corte.max()}")

filas_por_corte = sin_duplicados.groupby("Corte").size()
equipos_por_corte = sin_duplicados.groupby("Corte").Codigo_EQU.nunique()
mediana = equipos_por_corte.median()
atipicos = equipos_por_corte[(equipos_por_corte < 0.5 * mediana)]
display(pd.DataFrame({"filas": filas_por_corte, "equipos": equipos_por_corte}).describe().round(1).T)
registrar_hipotesis(
    "H07", "La cobertura de equipos es estable entre cortes",
    "se cumple" if atipicos.empty else "no se cumple",
    f"mediana {mediana:.0f} equipos por corte; cortes con menos de la mitad: "
    + ", ".join(f"{k} ({v})" for k, v in atipicos.items()),
)


H06 | SE CUMPLE | 44 cortes entre 202301 y 202608


,count,mean,std,min,25%,50%,75%,max
filas,44.0,295.6,188.6,22.0,199.2,304.5,365.0,1292.0
equipos,44.0,65.8,17.1,20.0,57.8,64.5,69.0,118.0


H07 | NO SE CUMPLE | mediana 64 equipos por corte; cortes con menos de la mitad: 202510 (20)


## H08. Cada código de equipo tiene un único nombre


In [8]:
nombres = sin_duplicados.dropna(subset=["Codigo_EQU", "EQU"])
codigos_multinombre = nombres.groupby(nombres.Codigo_EQU.str.upper()).EQU.nunique().gt(1).sum()
registrar_hipotesis(
    "H08", "Cada código de equipo tiene un único nombre",
    "se cumple" if codigos_multinombre == 0 else "no se cumple",
    f"{codigos_multinombre} códigos con más de un nombre; el nombre falta en {query.EQU.isna().mean():.0%} de filas pero es recuperable por código",
)


H08 | SE CUMPLE | 0 códigos con más de un nombre; el nombre falta en 41% de filas pero es recuperable por código


## Resumen de hipótesis


In [9]:
display(pd.DataFrame(HIPOTESIS).set_index("hipotesis"))


,enunciado,resultado,evidencia
hipotesis,,,
H01,Las columnas de query no tienen valores nulos,no se cumple,Codigo_EQU: 36 (0.2%); EQU: 6185 (40.7%); Resultado: 462 (3.0%); Meta: 308 (...
H02,No hay filas duplicadas,no se cumple,"2182 filas exactas repetidas (14.4%); cortes con más casos: 202502: 1223, 20..."
H03,Cada fila es única por Corte × Codigo_EQU × Indicador,no se cumple,247 llaves con valores distintos (1292 filas); 202502 trae Percepción/Adopci...
H04,Codigo_EQU sigue el formato AAA00000,no se cumple,"325 filas fuera de formato; ejemplos: Equ00074, Equ00052, Equ00026, Equ00028"
H05,"Resultado, Meta y Cumplimiento son numéricos",se cumple,"valores no numéricos: {'Resultado': 0, 'Meta': 0, 'Cumplimiento': 0}; los ra..."
H06,Corte tiene formato YYYYMM,se cumple,44 cortes entre 202301 y 202608
H07,La cobertura de equipos es estable entre cortes,no se cumple,mediana 64 equipos por corte; cortes con menos de la mitad: 202510 (20)
H08,Cada código de equipo tiene un único nombre,se cumple,0 códigos con más de un nombre; el nombre falta en 41% de filas pero es recu...


**Respuesta 1.1 (grano):** cada fila es la medición mensual de un indicador para un equipo; la identifican `Corte × Codigo_EQU × Indicador` (el `Frente` depende del indicador y `EQU` del código).
La llave no se cumple tal como viene por duplicados exactos, respuestas de encuesta sin agregar y códigos con formato distinto. El contraste con los catálogos está en `1_2` (indicadores) y `1_3` (equipos y entornos).
